# 🎮 Game 1: Finding the Optimal Landing Zone

## 🚀 Part A: Manual Exploration

You're piloting a lunar probe trying to land in a crater. Your mission: find the **safest landing spot** - the position with the lowest altitude.

### The Crater Surface
The crater follows this mathematical shape:

**L(w) = w² - 6w + 10**

Where:
- **w** = your horizontal landing position
- **L(w)** = the altitude (height) of the crater surface at position w
- **Lower altitude = safer landing!**

### Your Challenge
Try different landing positions and find the spot with the **minimum altitude**. There's one perfect spot somewhere... can you find it?

**🎯 Goal:** Discover that manual search is tedious and inefficient.

In [2]:
#@title 🔧 Game 1A Setup Code (click ▶ to run, you can hide this) { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from ipywidgets import Button, VBox, HBox, Output, HTML, Text
from IPython.display import display, clear_output

class Game1A_MoonLanding:
    def __init__(self):
        # Define the Crater (Loss Function)
        self.L = lambda w: w**2 - 6*w + 10
        self.optimal_w = 3.0
        self.optimal_loss = self.L(self.optimal_w)

        # Game state
        self.attempts = []
        self.best_attempt = None

        # Setup plot range
        self.w_range = np.linspace(-2, 8, 200)
        self.loss_range = self.L(self.w_range)

        # Create output widget
        self.output = Output()

    def create_game_ui(self):
        """Create the interactive UI"""

        # Text input for position with modern styling
        self.position_input = Text(
            value='0.0',
            description='Position (w):',
            style={'description_width': '100px'},
            layout={'width': '300px', 'height': '40px'}
        )

        # Modern styled buttons with better design
        land_button = Button(
            description='▶ Launch',
            button_style='',
            layout={'width': '120px', 'height': '40px'}
        )
        land_button.style.button_color = '#3b82f6'
        land_button.style.text_color = 'white'
        land_button.style.font_weight = 'bold'
        land_button.on_click(self.land_probe)

        reset_button = Button(
            description='↻ Reset',
            button_style='',
            layout={'width': '120px', 'height': '40px'}
        )
        reset_button.style.button_color = '#64748b'
        reset_button.style.text_color = 'white'
        reset_button.style.font_weight = 'bold'
        reset_button.on_click(self.reset_game)

        # Minimal stats display
        self.stats_display = HTML("""
        <div style='background: #1e293b; padding: 12px 20px; border-radius: 8px;
                    border: 1px solid #475569; display: inline-block;'>
            <span style='color: #94a3b8; font-size: 14px; margin-right: 20px;'>
                <strong style='color: #e2e8f0;'>Attempts:</strong> 0
            </span>
            <span style='color: #94a3b8; font-size: 14px; margin-right: 20px;'>
                <strong style='color: #e2e8f0;'>Best:</strong> --
            </span>
            <span style='color: #94a3b8; font-size: 14px;'>
                <strong style='color: #e2e8f0;'>Status:</strong> Searching...
            </span>
        </div>
        """)

        # Layout
        control_box = HBox([self.position_input, land_button, reset_button],
                          layout={'margin': '10px 0', 'align_items': 'center'})

        ui = VBox([
            self.output,
            control_box,
            self.stats_display
        ])

        display(ui)

        # Initial plot
        self.plot_crater()

    def land_probe(self, b):
        """Handle landing attempt"""
        try:
            guess = float(self.position_input.value)
        except ValueError:
            with self.output:
                clear_output(wait=True)
                print("⚠️ Please enter a valid number")
            return

        loss = self.L(guess)

        # Record attempt
        self.attempts.append((guess, loss))

        # Update best attempt
        if self.best_attempt is None or loss < self.best_attempt[1]:
            self.best_attempt = (guess, loss)

        # Update plot
        self.plot_crater()

        # Update stats
        self.update_stats()

    def plot_crater(self):
        """Plot the crater and all landing attempts with lunar theme"""
        with self.output:
            clear_output(wait=True)

            # Create figure with dark space background
            fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            # Determine y-axis limits based on attempts
            y_max = max(self.loss_range) + 5
            if self.attempts:
                max_attempt_loss = max(l for w, l in self.attempts if -2.5 <= w <= 8.5)
                y_max = min(max(max_attempt_loss * 1.2, max(self.loss_range) + 5), 100)

            # Plot crater surface with lunar color
            ax.plot(self.w_range, self.loss_range,
                   label='Crater Surface', color='#8b8b8b',
                   linestyle='-', linewidth=3.5, zorder=1)

            # Fill crater with moon-like gradient
            ax.fill_between(self.w_range, self.loss_range,
                           -2, alpha=0.3, color='#3a3a3a')

            # Add stars in background
            np.random.seed(42)
            star_x = np.random.uniform(-2.5, 8.5, 50)
            star_y = np.random.uniform(y_max * 0.7, y_max, 50)
            ax.scatter(star_x, star_y, c='white', s=np.random.uniform(1, 15, 50),
                      alpha=np.random.uniform(0.3, 0.9, 50), marker='*', zorder=0)

            # Plot all previous attempts (faded trails)
            if len(self.attempts) > 1:
                for i, (w, l) in enumerate(self.attempts[:-1]):
                    if l <= y_max:
                        alpha = 0.3 + 0.4 * (i / len(self.attempts))
                        ax.scatter([w], [l], color='#fbbf24', s=100,
                                 alpha=alpha, zorder=2, edgecolors='#f59e0b', linewidths=1.5)

            # Plot current/latest attempt
            if self.attempts:
                current_w, current_l = self.attempts[-1]

                if current_l <= y_max:
                    # Landing probe
                    ax.scatter([current_w], [current_l],
                              color='#ef4444', s=300, zorder=5,
                              label='Lunar Probe', marker='v',
                              edgecolors='#dc2626', linewidths=3)

                    # Add glow effect around probe
                    ax.scatter([current_w], [current_l],
                              color='#fecaca', s=600, zorder=4,
                              alpha=0.3, marker='o')

                    # Annotation
                    ax.annotate(f'L({current_w:.1f}) = {current_l:.2f}',
                               xy=(current_w, current_l),
                               xytext=(current_w+0.8, current_l+10),
                               arrowprops=dict(arrowstyle='->', color='#fbbf24', lw=2.5,
                                             connectionstyle='arc3,rad=0.3'),
                               fontsize=12, fontweight='bold',
                               color='white',
                               bbox=dict(boxstyle='round,pad=0.7',
                                       facecolor='#1e40af', alpha=0.9,
                                       edgecolor='#60a5fa', linewidth=2))
                else:
                    # Off-chart warning
                    ax.text(0.5, 0.5, f'PROBE OFF RADAR!\nPosition: w={current_w:.1f}\nAltitude: {current_l:.2f}',
                           transform=ax.transAxes,
                           fontsize=14, fontweight='bold',
                           color='#fbbf24',
                           ha='center', va='center',
                           bbox=dict(boxstyle='round,pad=1',
                                   facecolor='#7c2d12', alpha=0.9,
                                   edgecolor='#dc2626', linewidth=3))

                # Check if optimal
                if current_w == self.optimal_w:
                    ax.text(0.5, 0.92, 'PERFECT LANDING ACHIEVED',
                           transform=ax.transAxes,
                           fontsize=18, fontweight='bold',
                           color='#4ade80',
                           ha='center', va='top',
                           bbox=dict(boxstyle='round,pad=1',
                                   facecolor='#14532d', alpha=0.95,
                                   edgecolor='#22c55e', linewidth=3))

            # Mark the optimal point (when found)
            if self.best_attempt and self.best_attempt[0] == self.optimal_w:
                ax.scatter([self.optimal_w], [self.optimal_loss],
                          color='#fbbf24', s=500, zorder=4,
                          marker='*', label='Optimal Zone',
                          edgecolors='#f59e0b', linewidths=3)

                # Add glow effect
                for size, alpha in [(800, 0.1), (1200, 0.05)]:
                    ax.scatter([self.optimal_w], [self.optimal_loss],
                              color='#fef3c7', s=size, zorder=3,
                              alpha=alpha, marker='o')

            # Styling
            ax.set_title("LUNAR CRATER VISUALIZATION",
                        fontsize=20, fontweight='bold', pad=20,
                        color='#e0e7ff',
                        style='italic')
            ax.set_xlabel("Horizontal Position (w)", fontsize=14, fontweight='bold',
                         color='#c7d2fe')
            ax.set_ylabel("Altitude / Loss L(w)", fontsize=14, fontweight='bold',
                         color='#c7d2fe')
            ax.grid(True, alpha=0.15, linestyle='--', linewidth=1, color='#4a5568')
            ax.legend(fontsize=12, loc='upper right',
                     facecolor='#1a1a2e', edgecolor='#4a4a6a',
                     labelcolor='#e0e7ff', framealpha=0.9)
            ax.set_xlim(-2.5, 8.5)
            ax.set_ylim(-2, y_max)

            # Style the spines
            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)

            # Color the ticks
            ax.tick_params(colors='#a5b4fc', which='both')

            # Add landing target line
            ax.axhline(y=1, color='#22c55e', linestyle=':',
                      alpha=0.4, linewidth=2.5, label='Target Altitude')

            plt.tight_layout()
            plt.show()

    def update_stats(self):
        """Update the statistics display"""
        attempts_count = len(self.attempts)

        if self.best_attempt:
            best_w, best_l = self.best_attempt
            best_str = f"w={best_w:.1f}"

            # Check if found optimal
            if best_w == self.optimal_w:
                status_str = "✓ Found!"
                bg_color = '#14532d'
                border_color = '#22c55e'
            else:
                status_str = "Searching..."
                bg_color = '#1e293b'
                border_color = '#475569'
        else:
            best_str = "--"
            status_str = "Searching..."
            bg_color = '#1e293b'
            border_color = '#475569'

        self.stats_display.value = f"""
        <div style='background: {bg_color}; padding: 12px 20px; border-radius: 8px;
                    border: 1px solid {border_color}; display: inline-block;'>
            <span style='color: #94a3b8; font-size: 14px; margin-right: 20px;'>
                <strong style='color: #e2e8f0;'>Attempts:</strong> {attempts_count}
            </span>
            <span style='color: #94a3b8; font-size: 14px; margin-right: 20px;'>
                <strong style='color: #e2e8f0;'>Best:</strong> {best_str}
            </span>
            <span style='color: #94a3b8; font-size: 14px;'>
                <strong style='color: #e2e8f0;'>Status:</strong> {status_str}
            </span>
        </div>
        """

    def reset_game(self, b):
        """Reset the game state"""
        self.attempts = []
        self.best_attempt = None
        self.position_input.value = '0.0'
        self.update_stats()
        self.plot_crater()

print("✅ Game 1A setup complete! Run the next cell to start exploring.")

✅ Game 1A setup complete! Run the next cell to start exploring.


In [3]:
#@title ▶️ Launch Game 1A: Manual Landing { display-mode: "form" }
#@markdown Click the ▶ button and try different positions to find the minimum!

game1a = Game1A_MoonLanding()
game1a.create_game_ui()

## 🤔 Reflection: Part A

Take a moment to think about your experience:

### Question 1: The Search Process
**How many attempts did it take to find the optimal position?**

Was it quick and easy, or slow and frustrating? Now imagine doing this with **thousands** of parameters instead of just one - that's what real neural networks face!

### Question 2: Your Strategy
**What approach did you use?**

- Random guessing and checking?
- Systematic search (trying 1, 2, 3, 4... in order)?
- Smart adjustments (noticing patterns and adapting)?

Think about which strategy felt most efficient.

### Question 3: Certainty
**How confident are you that you found the TRUE minimum?**

- Are you absolutely certain it's the best possible position?
- Or did you just find a "good enough" spot and stop looking?
- How would you **prove** you found the global minimum?

---

### 💡 Key Insight

Manual search has serious problems:
- ❌ **Time-consuming**: Many attempts needed
- ❌ **Uncertain**: Hard to know if you found the true minimum
- ❌ **Doesn't scale**: Impossible with many parameters

**There must be a better way!** What if we had help? 🤔

Let's explore what happens when we have **perfect information**...

## 🚀 Part B: Learning from an Oracle

### The Scenario

Imagine **Mission Control** has advanced sensors that can detect the **exact optimal landing position** w*. They can tell you:

- Where you are now: **w_t**
- Where you should be: **w\***
- The difference: **w\* - w_t**

With this information, you can take **systematic steps** toward the optimal position!

### The Robbins-Monro Algorithm

This is one of the oldest optimization algorithms (1951). The update rule is beautifully simple:
```
w_t = w_{t-1} + α × (w* - w_{t-1})
```

Where:
- **w_{t-1}** = your current position
- **w\*** = the optimal position (from the oracle)
- **(w\* - w_{t-1})** = how far and which direction to move
- **α** = step size (how much of that distance to travel)

### Why This Matters

This shows: **if we knew where the optimum is, we could walk straight toward it systematically**. But in real ML problems, we don't have an oracle telling us w*!

**The big question:** How do we figure out which direction to move without an oracle? 🤔

*(This is what the rest of the games will teach you!)*

---

In [4]:
#@title 🔧 Game 1B Setup Code (Oracle-Guided) { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Button, VBox, HBox, Output, HTML, FloatText, IntText
from IPython.display import display, clear_output

class Game1B_OracleGuided:
    def __init__(self):
        # Loss function (same as Part A)
        self.L = lambda w: w**2 - 6*w + 10
        self.optimal_w = 3.0  # The oracle knows this!

        # Algorithm state
        self.current_w = 0.0
        self.history = []
        self.output = Output()

    def create_ui(self):
        """Create the interactive UI for oracle-guided learning"""

        # Input fields - used only on reset/start
        self.start_input = FloatText(
            value=0.0,
            description='Starting w:',
            style={'description_width': '100px'},
            layout={'width': '250px'}
        )

        self.alpha_input = FloatText(
            value=0.3,
            description='Step size α:',
            style={'description_width': '100px'},
            layout={'width': '250px'}
        )

        self.steps_input = IntText(
            value=5,
            description='Steps:',
            style={'description_width': '100px'},
            layout={'width': '250px'}
        )

        # Buttons
        reset_button = Button(
            description='🔄 Reset & Start',
            button_style='',
            layout={'width': '140px', 'height': '40px'}
        )
        reset_button.style.button_color = '#64748b'
        reset_button.style.text_color = 'white'
        reset_button.style.font_weight = 'bold'
        reset_button.on_click(self.reset)

        step_button = Button(
            description='➡️ Take Step',
            button_style='',
            layout={'width': '140px', 'height': '40px'}
        )
        step_button.style.button_color = '#3b82f6'
        step_button.style.text_color = 'white'
        step_button.style.font_weight = 'bold'
        step_button.on_click(self.take_steps)

        # Info display
        self.info_display = HTML("""
        <div style='background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;'>
            <p style='color: #94a3b8; margin: 5px 0; font-size: 14px; font-style: italic;'>
                ℹ️ Set your starting position and click "Reset & Start" to begin
            </p>
        </div>
        """)

        # Layout
        input_box = VBox([self.start_input, self.alpha_input, self.steps_input])
        button_box = HBox([reset_button, step_button], layout={'margin': '10px 0'})

        ui = VBox([
            self.output,
            input_box,
            button_box,
            self.info_display
        ])

        display(ui)

        # Show initial empty plot
        self.plot_initial()

    def plot_initial(self):
        """Show initial crater without any path"""
        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(14, 7), facecolor='#0a0e27')
            ax.set_facecolor('#1a1a2e')

            w_range = np.linspace(-1, 7, 200)
            loss_range = self.L(w_range)

            ax.plot(w_range, loss_range, color='#8b8b8b', linewidth=3, label='Crater Surface')
            ax.fill_between(w_range, loss_range, -2, alpha=0.3, color='#3a3a3a')

            # Show optimal point
            ax.scatter([self.optimal_w], [self.L(self.optimal_w)],
                       color='#22c55e', s=400, marker='*',
                       edgecolors='#16a34a', linewidths=3, zorder=6,
                       label='Optimal (Oracle)')

            ax.text(0.5, 0.5, 'Click "Reset & Start" to begin optimization',
                   transform=ax.transAxes,
                   fontsize=16, color='#94a3b8', ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=1', facecolor='#1e293b',
                            edgecolor='#475569', linewidth=2, alpha=0.9))

            ax.set_xlabel('Position (w)', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax.set_ylabel('Altitude L(w)', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax.set_title('Oracle-Guided Optimization', fontsize=14, color='#e0e7ff', fontweight='bold')
            ax.grid(True, alpha=0.15, color='#4a5568')
            ax.legend(facecolor='#1a1a2e', edgecolor='#4a4a6a', labelcolor='#e0e7ff', fontsize=10)
            ax.set_xlim(-1, 7)
            ax.set_ylim(-1, max(loss_range) + 2)

            for spine in ax.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax.tick_params(colors='#a5b4fc')

            plt.tight_layout()
            plt.show()

    def reset(self, b):
        """Reset the algorithm with the specified starting position"""
        # Get the starting position from input
        self.current_w = float(self.start_input.value)
        self.history = [(self.current_w, self.L(self.current_w))]

        # Update visualization and info
        self.plot_progress()
        self.update_info()

    def take_steps(self, b):
        """Take one or more Robbins-Monro steps"""
        # Check if algorithm has been started
        if not self.history:
            with self.output:
                clear_output(wait=True)
                print("⚠️ Please click 'Reset & Start' first to set your starting position!")
            return

        alpha = self.alpha_input.value
        num_steps = self.steps_input.value

        for _ in range(num_steps):
            # Robbins-Monro update: w_t = w_{t-1} + α × (w* - w_{t-1})
            direction = self.optimal_w - self.current_w
            self.current_w = self.current_w + alpha * direction
            self.history.append((self.current_w, self.L(self.current_w)))

        self.plot_progress()
        self.update_info()

    def plot_progress(self):
        """Plot the crater and convergence path"""
        with self.output:
            clear_output(wait=True)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0a0e27')

            # Left plot: Crater with path
            ax1.set_facecolor('#1a1a2e')
            w_range = np.linspace(-1, 7, 200)
            loss_range = self.L(w_range)

            ax1.plot(w_range, loss_range, color='#8b8b8b', linewidth=3, label='Crater Surface')
            ax1.fill_between(w_range, loss_range, -2, alpha=0.3, color='#3a3a3a')

            # Plot path with smart thinning for visualization
            if len(self.history) > 1:
                w_path = [h[0] for h in self.history]
                l_path = [h[1] for h in self.history]

                # If too many points, show path as line + key points
                if len(self.history) > 15:
                    # Show full path as thin line
                    ax1.plot(w_path, l_path, '-', color='#fbbf24',
                            linewidth=1.5, alpha=0.6, zorder=4)

                    # Show only start, end, and every 5th point
                    indices = [0] + list(range(4, len(w_path)-1, 5)) + [len(w_path)-1]
                    selected_w = [w_path[i] for i in indices]
                    selected_l = [l_path[i] for i in indices]

                    ax1.plot(selected_w, selected_l, 'o', color='#fbbf24',
                            markersize=6, zorder=5, label='Your Path')
                else:
                    # Show all points if not too many
                    ax1.plot(w_path, l_path, 'o-', color='#fbbf24', linewidth=2,
                            markersize=8, label='Your Path', zorder=5)

                # Highlight start position
                ax1.scatter([w_path[0]], [l_path[0]],
                          color='#60a5fa', s=250, marker='o',
                          edgecolors='#3b82f6', linewidths=3, zorder=6,
                          label='Start Position')

                # Current position
                ax1.scatter([w_path[-1]], [l_path[-1]],
                          color='#ef4444', s=300, marker='v',
                          edgecolors='#dc2626', linewidths=3, zorder=6,
                          label='Current Position')
            else:
                # Just starting - show only initial position
                ax1.scatter([self.history[0][0]], [self.history[0][1]],
                          color='#60a5fa', s=300, marker='o',
                          edgecolors='#3b82f6', linewidths=3, zorder=6,
                          label='Start Position')

            # Optimal point
            ax1.scatter([self.optimal_w], [self.L(self.optimal_w)],
                       color='#22c55e', s=400, marker='*',
                       edgecolors='#16a34a', linewidths=3, zorder=6,
                       label='Optimal (Oracle)')

            ax1.set_xlabel('Position (w)', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax1.set_ylabel('Altitude L(w)', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax1.set_title('Convergence Path on Crater Surface', fontsize=14, color='#e0e7ff', fontweight='bold')
            ax1.grid(True, alpha=0.15, color='#4a5568')
            ax1.legend(facecolor='#1a1a2e', edgecolor='#4a4a6a', labelcolor='#e0e7ff', fontsize=10)
            ax1.set_xlim(-1, 7)
            ax1.set_ylim(-1, max(loss_range) + 2)

            for spine in ax1.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax1.tick_params(colors='#a5b4fc')

            # Right plot: Convergence over steps
            ax2.set_facecolor('#1a1a2e')
            steps = list(range(len(self.history)))
            positions = [h[0] for h in self.history]

            # Plot with better styling
            ax2.plot(steps, positions, 'o-', color='#fbbf24',
                    linewidth=2, markersize=6 if len(steps) <= 20 else 4)

            # Optimal line
            ax2.axhline(y=self.optimal_w, color='#22c55e', linestyle='--',
                       linewidth=2.5, label=f'Optimal w* = {self.optimal_w}', alpha=0.8)

            # Add convergence region (within 1% of optimal)
            tolerance = 0.03
            ax2.axhspan(self.optimal_w - tolerance, self.optimal_w + tolerance,
                       color='#22c55e', alpha=0.1, label='Convergence Zone')

            ax2.set_xlabel('Step Number', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax2.set_ylabel('Position w', fontsize=12, color='#c7d2fe', fontweight='bold')
            ax2.set_title('Convergence Over Time', fontsize=14, color='#e0e7ff', fontweight='bold')
            ax2.grid(True, alpha=0.15, color='#4a5568')
            ax2.legend(facecolor='#1a1a2e', edgecolor='#4a4a6a', labelcolor='#e0e7ff', fontsize=10)

            for spine in ax2.spines.values():
                spine.set_edgecolor('#4a4a6a')
                spine.set_linewidth(2)
            ax2.tick_params(colors='#a5b4fc')

            plt.tight_layout()
            plt.show()

    def update_info(self):
        """Update the information display"""
        direction = self.optimal_w - self.current_w
        direction_text = f"Move right +{direction:.2f}" if direction > 0 else f"Move left {direction:.2f}"

        if abs(direction) < 0.01:
            direction_text = "✓ ARRIVED!"
            bg_color = '#14532d'
            border_color = '#22c55e'
        else:
            bg_color = '#1e293b'
            border_color = '#475569'

        self.info_display.value = f"""
        <div style='background: {bg_color}; padding: 15px; border-radius: 8px; border: 1px solid {border_color};'>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Current Position:</strong> w = {self.current_w:.4f}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Oracle Says:</strong> w* = {self.optimal_w:.2f}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Direction:</strong> {direction_text}</p>
            <p style='color: #e2e8f0; margin: 5px 0; font-size: 15px;'><strong>Steps Taken:</strong> {len(self.history) - 1}</p>
        </div>
        """

print("✅ Game 1B setup complete! Run the next cell to explore oracle-guided learning.")

✅ Game 1B setup complete! Run the next cell to explore oracle-guided learning.


In [5]:
#@title ▶️ Launch Game 1B: Oracle-Guided Learning { display-mode: "form" }
#@markdown Try different step sizes (α) and see how it affects convergence!

game1b = Game1B_OracleGuided()
game1b.create_ui()

## 🤔 Reflection: Part B (Oracle-Guided)

### Experiments to Try

1. **Different step sizes (α):**
   - α = 0.01: What happens? (Slow but steady)
   - α = 0.1: Seems good!
   - α = 0.5: Still works, but different behavior
   - α = 1.0: What happens at α = 1? (Instant convergence!)
   - α = 1.5: What about α > 1? (Overshooting!)

2. **Different starting positions:**
   - Start at w = -2, 0, 5, 7
   - Does the algorithm always converge?

### Questions to Consider

**Question 1: The Power of Perfect Information**

With the oracle telling you w*, convergence is **guaranteed** and **systematic**. How many steps does it take compared to manual search?

**Question 2: The Update Rule**

Look at the formula: `w_t = w_{t-1} + α × (w* - w_{t-1})`

- What does `(w* - w_{t-1})` represent? (The direction and distance to the optimum)
- What role does α play? (Controls step size)
- Why does α = 1.0 work in one step? (Takes the full distance!)

**Question 3: The Critical Problem**

This algorithm is elegant and works perfectly... **but**:

### 💡 **THE BIG PROBLEM**

**In real machine learning, we don't have an oracle!**

We don't know w* in advance. If we did, we wouldn't need to optimize - we'd just set w = w* immediately!

**The Challenge:**
- ✅ Part A: Manual search is tedious
- ✅ Part B: Oracle-guided is perfect... but impossible in practice
- ❓ **What's the solution?**

We need to figure out **which direction to move** using only **local information** - things we can actually compute!

---

### 🎓 Key Takeaways from Game 1

1. **Manual search doesn't scale** - Impossible with many parameters
2. **Oracle-guided learning is perfect** - But requires impossible knowledge (w*)
3. **We need local information** - Something we can calculate from L(w) itself

**The Core Question:** If we can't see the whole landscape and we don't have an oracle, what **local** information at position w could tell us which direction leads downhill?

💭 *Think about this: If you're standing in the crater with fog all around, what could you measure that would tell you which way is down?*